# 03 - Labels: DeepFace guess + manual review
**Stage:** the paper pre-annotates with DeepFace and then has humans revise the labels. Here you review every face,
write your fixes into the `CORRECTIONS` dict, and the final dataset is built.

| | |
|---|---|
| **Input** | `data/faces_auto.json` (notebook 02), `data/labels_corrections.json` (your earlier fixes) |
| **Output** | `data/dataset.json` (boxes + landmarks + final labels), `data/dataset_eval_only.json`, `data/labels_corrections.json` |
| **Code used** | `src/labeling.py` |

**Important for the extra photos:** DeepFace is noisy on real photos (it often says "sad"/"neutral"). Check every face in
the pictures below and add a correction for each wrong one. Face numbers are left to right and match the picture.

In [ ]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

In [ ]:
import cv2, matplotlib.pyplot as plt
from src import labeling
RECORDS = utils.load_json(config.FACES_JSON)

def review_image(rec):
    img = cv2.imread(rec["path"])
    t = max(2, round(max(img.shape[:2]) / 400))
    for f in rec["faces"]:
        x, y, w, h = f["box"]
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 200, 0), t)
        cv2.putText(img, f'{f["id"]}:{f["auto_emotion"]}', (x, max(18, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, max(0.6, t * 0.35), (0, 255, 255), t)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

PER_FIG = 6
for s in range(0, len(RECORDS), PER_FIG):
    fig, axes = plt.subplots(2, 3, figsize=(21, 10))
    for ax in axes.ravel(): ax.axis("off")
    for ax, rec in zip(axes.ravel(), RECORDS[s:s + PER_FIG]):
        ax.imshow(review_image(rec)); ax.set_title(f'{rec["image"]}  ({rec["n_faces"]} faces, {config.difficulty(rec["n_faces"])})', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# EDIT HERE: {image name: {face number: correct label}}. Allowed labels: angry, disgust, fear, happy, sad, surprise, neutral, ignore
# ("ignore" removes a wrongly detected face). Your earlier fixes are loaded automatically; add new ones below.
NEW_CORRECTIONS = {
    # "image25.jpg": {1: "happy", 3: "sad"},
}
DROP_IMAGES = []          # images to leave out completely
REVIEWED = True           # set to True only after you checked EVERY face above

CORRECTIONS, drop_saved = utils.load_corrections()
for img, fixes in NEW_CORRECTIONS.items():
    CORRECTIONS.setdefault(img, {}).update(fixes)
DROP_IMAGES = sorted(set(DROP_IMAGES) | set(drop_saved))
utils.save_json({**{i: {str(k): v for k, v in f.items()} for i, f in CORRECTIONS.items()}, "drop_images": DROP_IMAGES},
                config.CORRECTIONS_JSON)
for k in CORRECTIONS:
    if k not in {r["image"] for r in RECORDS}:
        print("WARNING: correction for unknown image name:", k)

In [ ]:
DATA, n_changed, n_faces_total = labeling.build_dataset(RECORDS, CORRECTIONS, DROP_IMAGES)
per_diff, emo = labeling.dataset_stats(DATA)
print(f"Images: {len(DATA)} | faces: {n_faces_total} | labels you changed: {n_changed} ({n_changed / max(1, n_faces_total):.1%}) | reviewed: {REVIEWED}")
print("\nImages / faces per difficulty (paper: Easy 76/174, Medium 34/171, Hard 9/114):")
print(per_diff.to_string())
print("\nEmotion distribution (faces):")
print(emo.to_string())
for d in config.DIFFICULTIES:
    if per_diff.loc[d, "images"] < 5:
        print(f"!! Only {per_diff.loc[d, 'images']} {d} images - results for {d} will be very uncertain.")
if n_faces_total and n_changed / n_faces_total < 0.10:
    print("note: you changed <10% of DeepFace labels - re-check the labels face by face.")

# Safety check: photos from the first study must give the same number of faces as before (the saved model answers depend on it)
if config.DATASET_EVAL_JSON.exists():
    old = {r["image"]: r["n_faces"] for r in utils.load_json(config.DATASET_EVAL_JSON)}
    new = {r["image"]: r["n_faces"] for r in DATA}
    changed = {i: (old[i], new[i]) for i in old if i in new and old[i] != new[i]}
    missing = [i for i in old if i not in new]
    if changed or missing:
        print("!! WARNING: these photos differ from the first study (old faces, new faces):", changed, "| missing:", missing)
        print("   Saved answers for them no longer match. Fix with 'ignore' corrections, or delete their lines from raw_outputs.jsonl.")
    else:
        print("OK: all photos from the first study have the same faces as before.")

utils.save_json(DATA, config.DATASET_JSON)
utils.save_json(labeling.eval_only_view(DATA), config.DATASET_EVAL_JSON)
print("saved:", config.DATASET_JSON.name, "and", config.DATASET_EVAL_JSON.name)

**Next:** `04_visual_prompts.ipynb`